# 12.10 怎樣判斷回答真的受聲音影響？


四題單音為 200、220、440、660 Hz，規則仍是高於 300 Hz 回 high。答案依序為 low、low、high、high。假如完全不聽，每題都回 high，也能對 2/4；有分數不等於有使用聲音。


In [1]:
# @title 準備本節的工具（首次執行）
from pathlib import Path
import os
import subprocess
import sys

# Colab 會先下載專案；本機 Jupyter 沿用已開啟的 repo。
try:
    import google.colab
except ImportError:
    in_colab = False
else:
    in_colab = True

if in_colab:
    root = Path("/content/tiny-perceptron-vlm")
    if not root.exists():
        subprocess.run(
            ["git", "clone", "--depth", "1", "https://github.com/birdhackor/tiny-perceptron-vlm.git", str(root)],
            env={**os.environ, "GIT_LFS_SKIP_SMUDGE": "1"},
            check=True,
        )
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "-e", str(root)], check=True)
    os.chdir(root)

root = Path.cwd().resolve()
while not (root / "pyproject.toml").exists() and root != root.parent:
    root = root.parent
if not (root / "tiny_perceptron").is_dir():
    raise RuntimeError("本機請先依 course/first-steps.md 開啟 repo；免安裝練習可使用本節的 Colab 入口")
sys.path.insert(0, str(root))
import torch

torch.set_num_threads(1)
torch.manual_seed(42)

In [2]:
frequencies = [200, 220, 440, 660]
truth = ["high" if f > 300 else "low" for f in frequencies]
no_audio = ["high"] * len(frequencies)
accuracy = sum(a == b for a, b in zip(truth, no_audio)) / len(truth)
print("頻率", frequencies)
print("標準答案", truth)
print("不聽聲音的固定猜測", accuracy)

頻率 [200, 220, 440, 660]
標準答案 ['low', 'low', 'high', 'high']
不聽聲音的固定猜測 0.5


這個固定答案對照印出 0.5。測試集若高低音不平衡，基準也跟著變。既有 14 題中低音 8 題、高音 6 題，固定回 low 可對 8/14；必須報分母與各類數量，才能理解模型的 11/14。

接著固定提問，替換聲音。例如 200 Hz 應回 low，換成 310 Hz 應回 high。替換後要用新聲音的答案判正確；若繼續拿舊答案比，看到答覆改變只能說它受替換影響，不能直接說答錯。

既有試驗在原聲音對 11/14，全部改成靜音時對 6/14、都回 high。靜音沒有可判斷的單音頻率，所以這裡的分數只是與原題答案相比的控制，不是「靜音本應 high」的規則。

再交換高低來源，按舊答案得 3/14，按新聲音答案仍 11/14。這支持聲音被利用，而且說明答案變化與正確性要分開看；替換的 14 題有 2 題新舊標籤未變，不能要求所有題都一定翻轉。它仍只涉及合成單音，不推出真人需求辨識。

練習若十題有九題 low，固定回 low 的基準是多少？再列出「原音、替音按新答案、靜音」三個控制各能回答什麼。安排訓練組與留出檢查組時，以同一原聲音及由它製作的變體為一組；原聲音若放留出，變體也都留出，避免把練習音訊的改版當作陌生聲音。

<details>
<summary>回顧與查證</summary>

可回顧：[12.9聲音訓練接口](https://birdhackor.github.io/tiny-perceptron-vlm/12.9.html)、[11.8替圖檢驗](https://birdhackor.github.io/tiny-perceptron-vlm/11.8.html)。

原實驗的資料切分、更新設定與逐題結果見[既有實報](https://github.com/birdhackor/tiny-perceptron-vlm/blob/main/docs/course-experiments/results/audio.json)；重做入口見[實驗說明](https://github.com/birdhackor/tiny-perceptron-vlm/blob/main/docs/course-experiments/README.md)。這是上述有限任務的歷史紀錄。

</details>
